### Notebook for calculating correlations between MBS accum and SAM indices and reconstructions

In [1]:
import xarray as xr
import pandas as pd

from scipy.ndimage import gaussian_filter1d

import numpy as np

from scipy.stats import pearsonr
from scipy.stats import t
import scipy.stats as stats

In [2]:
# read in MBS accum dataset
mbs_accum = pd.read_csv("mbs_accum_strainrate.csv")
mbs_accum_df = mbs_accum[["Year", "IE_accum_SR2.59"]]
mbs_accum_df = mbs_accum_df.rename(columns={"IE_accum_SR2.59": "IE_accum"})
mbs_accum_df

,Year,IE_accum
0,2017,0.284600
1,2016,0.225253
2,2015,0.339336
3,2014,0.157400
4,2013,0.217896
...,...,...
1140,877,0.513151
1141,876,0.377573
1142,875,0.411272
1143,874,0.244677


In [5]:
# read in SAM datwyler reconstruction
sam_datwyler = pd.read_csv("data/Best_SAM_reconstruction_datwyler.csv")
sam_recon = sam_datwyler[["Year", "Final_DJF_SAM_reconstruction"]]
sam_recon=sam_recon.rename(columns={"Final_DJF_SAM_reconstruction": "SAM_DJF_datwyler"})
sam_recon

,Year,SAM_DJF_datwyler
0,1000,-3.9471
1,1001,-1.1801
2,1002,0.8654
3,1003,0.8499
4,1004,0.0414
...,...,...
1001,2001,0.9914
1002,2002,0.4824
1003,2003,1.5419
1004,2004,1.6104


In [6]:
# import the reconstruction nc file from king et al
nc_file = 'data/reconstruction_king.nc'
king_dset = xr.open_dataset(nc_file)
king_dset

<xarray.Dataset>
Dimensions:          (years: 2000, percentiles: 99)
Coordinates:
  * years            (years) float64 1.0 2.0 3.0 ... 1.998e+03 1.999e+03 2e+03
  * percentiles      (percentiles) float64 1.0 2.0 3.0 4.0 ... 97.0 98.0 99.0
Data variables:
    sam              (years) float64 ...
    sam_sigma        (years) float64 ...
    sam_percentiles  (percentiles, years) float64 ...

In [7]:
# select just percentiles =50
king_dset_50 = king_dset.sel(percentiles = 50.0)
king_df = king_dset_50.to_dataframe().reset_index()
king_df = king_df.rename(columns = {"years" : "Year", "sam" : "sam_king"})
king_df

,Year,percentiles,sam_king,sam_sigma,sam_percentiles
0,1.0,50.0,-0.151025,2.796674,0.317618
1,2.0,50.0,1.447257,2.640292,0.278802
2,3.0,50.0,1.468665,2.640292,0.278802
3,4.0,50.0,1.357770,2.640292,0.278802
4,5.0,50.0,1.884448,2.640292,0.278802
...,...,...,...,...,...
1995,1996.0,50.0,0.075497,1.527640,0.125431
1996,1997.0,50.0,1.112353,1.538503,0.125639
1997,1998.0,50.0,0.854209,1.579861,0.127078
1998,1999.0,50.0,2.666772,1.523103,0.121713


In [19]:
# import abram sam reconstruction
abram_sam = pd.read_csv("data/sam_abram.csv")
abram_sam

,age_CE,SAM_Index,SAM_Index_err,RE,RE_crit,SAM_Index_70yloess,SAM_Index_+error_70yloess,SAM_Index_-error_70yloess,Marshall_SAM_Index
0,2007,-2.655296,3.217976,-0.015027,-0.237163,1.815786,4.752763,-1.119229,-0.083
1,2006,2.516332,3.217976,-0.015027,-0.237163,1.762162,4.682995,-1.156849,2.197
2,2005,0.250082,3.236963,-0.027041,-0.115012,1.708567,4.613422,-1.194603,0.987
3,2004,2.043958,3.236963,-0.027041,-0.115012,1.655018,4.544066,-1.232481,1.857
4,2003,0.834730,3.236963,-0.027041,-0.115012,1.601538,4.474956,-1.270464,1.037
...,...,...,...,...,...,...,...,...,...
1003,1004,-1.813489,3.011280,0.111179,0.006600,-1.579751,1.431294,-4.590993,NaN
1004,1003,-1.529136,3.011280,0.111179,0.006600,-1.563468,1.447598,-4.574714,NaN
1005,1002,-1.007336,3.011280,0.111179,0.006600,-1.548646,1.462439,-4.559895,NaN
1006,1001,-1.523829,3.011280,0.111179,0.006600,-1.535509,1.475590,-4.546760,NaN


In [20]:
abram_sam = abram_sam.rename(columns = {"age_CE" : "Year", "SAM_Index" : "SAM_abram"})
abram_sam = abram_sam[["Year", "SAM_abram"]]
abram_sam

,Year,SAM_abram
0,2007,-2.655296
1,2006,2.516332
2,2005,0.250082
3,2004,2.043958
4,2003,0.834730
...,...,...
1003,1004,-1.813489
1004,1003,-1.529136
1005,1002,-1.007336
1006,1001,-1.523829


In [10]:
# import marshal sam index
sam_marshall = pd.read_csv("data/sam_marshall.csv")
sam_marshall

,Year,ANN,AUT,WIN,SPR,SUM
0,1957,-4.24,-2.52,-0.68,-3.55,-2.52
1,1958,-1.27,-2.97,-0.92,1.42,1.12
2,1959,0.47,-0.94,-1.22,3.20,0.31
3,1960,1.52,0.74,0.32,1.00,-0.08
4,1961,1.22,0.48,0.93,0.85,2.19
...,...,...,...,...,...,...
64,2021,3.03,0.34,0.57,2.80,3.25
65,2022,4.18,1.93,0.03,4.25,3.83
66,2023,2.04,0.74,-0.11,0.25,3.51
67,2024,1.63,2.07,-1.21,1.81,-0.31


In [11]:
# extract fogt column from datwyler df
sam_fogt = sam_datwyler[["Year", "Target SAM (Fogt)"]].dropna()
sam_fogt = sam_fogt.rename(columns={"Target SAM (Fogt)":"SAM_fogt"})
sam_fogt

,Year,SAM_fogt
866,1866,0.3444
867,1867,1.1934
868,1868,-2.4186
869,1869,-1.4676
870,1870,-2.1726
...,...,...
1001,2001,0.2834
1002,2002,1.2714
1003,2003,1.9134
1004,2004,0.3954


In [12]:
def lag1_autocorr(series):
    # calculate lag1 autocorrelation
    return np.corrcoef(series[:-1], series[1:])[0,1]


def n_eff(N, p1, p2):
    # calculates effective degrees of freedom
    # p1 and p2 are the lag 1 autocorrelations for the two time series
    neff = N*((1-p1*p2)/(1+p1*p2))
    return neff

def corr_pvalue_neff(x, y):
    """calculates the Pearson correlation (r) between two time series
    returns r, p, and effective degrees of freedom. p is adjusted based on the effective degrees of freedom
    """
    
    # calculate correlation between two variables
    r = np.corrcoef(x, y)[0,1]

    # calculate lag1 autocorrelations

    rho_x = lag1_autocorr(x)
    rho_y = lag1_autocorr(y)

    # calculate effective sample size
    N = len(x)
    N_eff = n_eff(N, rho_x, rho_y)

    # this catches cases where N_eff ends up too small 
    if N_eff <= 2:
        print("N_eff is too small :(")
        return r, np.nan, N_eff   # not enough DOF


    # degrees of freedom
    dof = N_eff -2

    # calculate t-statistic
    t_stat = r * np.sqrt(dof / (1 - r**2))

    # calculate p-value (two sided)
    p = 2 * (1 - t.cdf(abs(t_stat), df = dof))

    return r, p, N_eff

In [52]:
def calc_correlations(df1, df2, col1, col2, sigma=2):

    merged = pd.merge(
        df1[['Year', col1]],
        df2[['Year', col2]].dropna(),
        on='Year',
        how='inner'
    ).sort_values('Year')

    r_raw, p_raw, N_eff_raw = corr_pvalue_neff(merged[col1], merged[col2])

    merged[f'{col1}_smooth'] = gaussian_filter1d(
        merged[col1].values,
        sigma=sigma
    )

    merged[f'{col2}_smooth'] = gaussian_filter1d(
        merged[col2].values,
        sigma=sigma
    )


    # smoothed corr
    r_smooth, p_smooth, N_eff_smooth= corr_pvalue_neff( merged[f'{col1}_smooth'],  merged[f'{col2}_smooth'])

    return {
        'Data' : col2,
        'raw_corr': r_raw,
        'p_raw' :p_raw,
        'N_eff_raw': N_eff_raw,
        'sigma': sigma,
        'smooth_corr': r_smooth,
        'p_smooth': p_smooth,
        'N_eff_smooth' : N_eff_smooth,
        'n_years': len(merged)
    }

In [54]:
calc_correlations(mbs_accum_df, sam_recon, "IE_accum", "SAM_DJF_datwyler", sigma=1.5)

{'Data': 'SAM_DJF_datwyler',
 'raw_corr': 0.05802436159456324,
 'p_raw': 0.08124293410947692,
 'N_eff_raw': 903.8647112620624,
 'sigma': 1.5,
 'smooth_corr': 0.11610002378002231,
 'p_smooth': 0.4082637071148123,
 'N_eff_smooth': 52.8880126631325,
 'n_years': 1006}

In [55]:
# king correlations
calc_correlations(mbs_accum_df, king_df, "IE_accum", "sam_king", sigma=1.5)

{'Data': 'sam_king',
 'raw_corr': 0.0121475153174572,
 'p_raw': 0.6981583310786239,
 'N_eff_raw': 1021.6213340582522,
 'sigma': 1.5,
 'smooth_corr': -0.04492406772797983,
 'p_smooth': 0.7179174322968742,
 'N_eff_smooth': 67.09363605573829,
 'n_years': 1128}

In [56]:
# abram correlations
calc_correlations(mbs_accum_df, abram_sam, "IE_accum", "SAM_abram", sigma=1.5)

{'Data': 'SAM_abram',
 'raw_corr': -0.028961129185479648,
 'p_raw': 0.38279483697887606,
 'N_eff_raw': 910.2676643409391,
 'sigma': 1.5,
 'smooth_corr': -0.08449692030719194,
 'p_smooth': 0.5397762617614825,
 'N_eff_smooth': 54.96581709185915,
 'n_years': 1008}

In [58]:
# masrhsall correlations
calc_correlations(mbs_accum_df, sam_marshall, "IE_accum", "ANN", sigma=1.5)

{'Data': 'ANN',
 'raw_corr': -0.007259386898440764,
 'p_raw': 0.9552545091084312,
 'N_eff_raw': 62.237568336706815,
 'sigma': 1.5,
 'smooth_corr': -0.1769542432177978,
 'p_smooth': 0.8524604120630692,
 'N_eff_smooth': 3.4913930642627635,
 'n_years': 61}

In [60]:
# marshall correlations
calc_correlations(mbs_accum_df, sam_marshall, "IE_accum", "SUM", sigma=1.5)

{'Data': 'SUM',
 'raw_corr': -0.07056561574674194,
 'p_raw': 0.5856431158843871,
 'N_eff_raw': 62.03360338385663,
 'sigma': 1.5,
 'smooth_corr': -0.08999556328008237,
 'p_smooth': 0.9137095229022496,
 'N_eff_smooth': 3.867731635380814,
 'n_years': 61}

In [61]:
# fogt correlations
calc_correlations(mbs_accum_df, sam_fogt, "IE_accum", "SAM_fogt", sigma=1.5)

{'Data': 'SAM_fogt',
 'raw_corr': 0.04291094021284804,
 'p_raw': 0.6174784247374254,
 'N_eff_raw': 137.83341808234096,
 'sigma': 1.5,
 'smooth_corr': 0.17258095058308795,
 'p_smooth': 0.5953096016301389,
 'N_eff_smooth': 11.816060285153759,
 'n_years': 140}